In [181]:
import torch
import torch.nn as nn
import os
import json
import pandas as pd
from pprint import pprint
from models.queryner_teacher import QueryNERTeacher
from models.crf_layer import CRFOutputLayer
from models.base_model import BaseNERModel
from models.distilbert_student import DistilBERTStudent
from models.tinybert_student import TinyBERTStudent
from models.bilstm_student import BiLSTMStudent
from models.ner_dataset import NERDataset
from models.utils import load_label_info, create_dataloaders

from seqeval.metrics import classification_report
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from seqeval.scheme import IOB1, IOB2

In [182]:
def load_json_to_df(json_path):
    with open(json_path) as file:
        data = json.load(file)
    df = pd.DataFrame(data=data["successful_experiments"])
    df.insert(0, "id", df.index + 1)
    return df

In [183]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")

tokenizer, train_loader, val_loader, test_loader = create_dataloaders(
    train_path=r"D:\Dafa\Project\queryner-kd\data\processed\train.json",
    val_path=r"D:\Dafa\Project\queryner-kd\data\processed\validation.json",
    test_path=r"D:\Dafa\Project\queryner-kd\data\processed\test.json",
    model_name="bert-base-uncased",
    batch_size=16,
    max_length=128
)

cuda


## Run Inference All Models

In [205]:
models_name = [
    'bilstm-crf.pt',
                'bilstm-nocrf.pt',
                'distilbert-crf.pt',
                'distilbert-nocrf.pt',
                # 'teacher-crf.pt',
                # 'teacher-nocrf.pt',
                'tinybert-crf.pt',
                'tinybert-nocrf.pt']

model_folder_path = r"D:\Dafa\Project\queryner-kd\checkpoints\models-baseline"

In [206]:
def instantiate_student(model_class, model_path, label_info, use_crf, device):
    if model_class == "DistilBERTStudent":
        model = DistilBERTStudent(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf,
        )
    elif model_class == "TinyBertStudent":
        model = TinyBERTStudent(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf,
        )
    elif model_class == "BiLSTMStudent":
        model = BiLSTMStudent(
            num_labels=35,
            use_crf=use_crf,
            model_name_for_vocab=model_path,
            label_info=label_info,
            emb_dim=768
        )
    else:
        raise ValueError(model_class)
    return model.to(device)


def instantiate_teacher(model_class, model_path, label_info, use_crf, device):
    if model_class == "QueryNERTeacher":
        model = QueryNERTeacher(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf,
        )
    else:
        raise ValueError(model_class)
    return model.to(device)


def load_student_model(checkpoint_path, model_class, model_path, label_info, use_crf, device):
    model = instantiate_student(model_class, model_path, label_info, use_crf, device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.eval()
    return model

def load_teacher_model(checkpoint_path, model_class, model_path, label_info, use_crf, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    teacher = instantiate_teacher(model_class, model_path, label_info, use_crf, device)
    teacher.load_state_dict(checkpoint["model_state_dict"])
    teacher.to(device)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher

def load_all_models(models_name, model_folder_path, label_info, device):
    loaded = {}
    for name in models_name:
        path = os.path.join(model_folder_path, name)
        if "teacher" in name:
            use_crf = "-crf" in name
            loaded[name.split(".")[0]] = load_teacher_model(
                path,
                "QueryNERTeacher",
                "bert-base-uncased",
                label_info,
                use_crf,
                device,
            )
        elif "distilbert" in name:
            use_crf = "-crf" in name
            loaded[name.split(".")[0]] = load_student_model(
                path,
                "DistilBERTStudent",
                "distilbert-base-uncased",
                label_info,
                use_crf,
                device,
            )
        elif "tinybert" in name:
            use_crf = "-crf" in name
            loaded[name.split(".")[0]] = load_student_model(
                path,
                "TinyBertStudent",
                "huawei-noah/TinyBERT_General_4L_312D",
                label_info,
                use_crf,
                device,
            )
        elif "bilstm" in name:
            use_crf = "-crf" in name
            loaded[name.split(".")[0]] = load_student_model(
                path,
                "BiLSTMStudent",
                "bert-base-uncased",
                label_info,
                use_crf,
                device
            )
    return loaded

In [207]:
models = load_all_models(models_name, model_folder_path, label_info, device)

In [208]:
def predict(model, encoding, device):
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    return outputs["pred"]

In [209]:
def decode_word_preds(pred_ids, word_ids):
    word_preds = []
    prev_word = None

    for wid, pid in zip(word_ids, pred_ids):

        if wid is None:
            continue

        if wid != prev_word:

            if isinstance(pid, torch.Tensor):
                pid = pid.item()

            word_preds.append(int(pid))

        prev_word = wid

    return word_preds

In [210]:
def run_inference_models(models, dataloader, device):
    results = []

    for batch in dataloader:

        batch_preds = {}

        for name, model in models.items():
            batch_preds[name] = predict(model, batch, device)

        for i in range(len(batch["tokens"])):

            tokens = batch["tokens"][i]
            gold = batch["ner_tags"][i]
            word_ids = batch["word_ids"][i]

            entry = {
                "tokens": tokens,
                "gold": gold
            }

            for name, preds in batch_preds.items():
                pred_ids = preds[i]
                entry[name] = decode_word_preds(pred_ids, word_ids)

            results.append(entry)

    return results

In [211]:
results = run_inference_models(models, test_loader, device)

In [ ]:
# with open(r"D:\Dafa\Project\queryner-kd\result-data\qualitative\baseline\qualitative_results_baseline.json", "w") as f:
#     json.dump(results, f, indent=4)

## Map Labels, Create Gold and Predicted Labels Lists

In [215]:
json_path = r"D:\Dafa\Project\queryner-kd\result-data\qualitative\baseline\qualitative_results_baseline.json"

with open(json_path, "r") as file:
    results = json.load(file)

In [216]:
def map_id_to_label(pred_ids, id2label):
    return [id2label[int(pid)] for pid in pred_ids]

models_name_without_pt = [name.split(".")[0] for name in models_name]
results_label = []

for entry in results:
    new_entry = {
        "tokens": entry["tokens"],
        "gold": map_id_to_label(entry["gold"], label_info["id2label"])
    }
    for model in models_name_without_pt:
        new_entry[model] = map_id_to_label(entry[model], label_info["id2label"])
    results_label.append(new_entry)

In [ ]:
# with open(r"D:\Dafa\Project\queryner-kd\result-data\qualitative\baseline\qualitative_results_label_baseline.json", "w") as file:
#     # results_label = json.load(file)
#     json.dump(results_label, file, indent=4)

In [220]:
gold_labels = [entry["gold"] for entry in results_label]
pred_labels = {model: [entry[model] for entry in results_label] for model in models_name_without_pt}

## Token-Level Evaluation

### Entity Based Acc, Precision, Recall, F1 Scores

In [221]:
entity_based_scores = {}

for model in models_name_without_pt:
    entity_based_scores[model] = {
        "accuracy_score": accuracy_score(gold_labels, pred_labels[model])*100,
        "precision": precision_score(gold_labels, pred_labels[model], mode="strict", scheme=IOB2)*100,
        "recall": recall_score(gold_labels, pred_labels[model], mode="strict", scheme=IOB2)*100,
        "f1": f1_score(gold_labels, pred_labels[model], mode="strict", scheme=IOB2)*100
    }

In [222]:
df_scores = pd.DataFrame.from_dict(entity_based_scores, orient="index")
df_scores = df_scores.reset_index().rename(columns={"index": "model"})
df_scores

,model,accuracy_score,precision,recall,f1
0,bilstm-crf,59.961686,57.082749,51.979566,54.411765
1,bilstm-nocrf,60.700602,54.026992,52.830992,53.422299
2,distilbert-crf,65.462507,61.597821,57.769264,59.622144
3,distilbert-nocrf,65.982485,60.714286,59.344402,60.021529
4,tinybert-crf,60.700602,55.439965,54.448702,54.939863
5,tinybert-nocrf,61.412151,55.536407,54.874415,55.203426


In [ ]:
# df_scores.to_csv(r"D:\Dafa\Project\queryner-kd\result-data\entity-based-eval\scores_no_scheme.csv", index=False)

### Check I- without preceding B-

In [150]:
def check_I_entity(gold_labels):
    i_without_b = 0
    for sentence in gold_labels:
        for i, label in enumerate(sentence):
            if label.startswith("I-"):
                if i == 0 or not sentence[i-1].endswith(label[2:]):
                    i_without_b += 1
    return i_without_b

In [151]:
i_without_b = {}

for model in models_name_without_pt:
    i_without_b[model] = check_I_entity(pred_labels[model])
    print(f"Total I- tags without preceding B- tags for {model}: {i_without_b[model]}")
    print("==" * 20)

Total I- tags without preceding B- tags for bilstm-crf: 84
Total I- tags without preceding B- tags for bilstm-nocrf: 213
Total I- tags without preceding B- tags for distilbert-crf: 119
Total I- tags without preceding B- tags for distilbert-nocrf: 96
Total I- tags without preceding B- tags for teacher-crf: 127
Total I- tags without preceding B- tags for teacher-nocrf: 139
Total I- tags without preceding B- tags for tinybert-crf: 177
Total I- tags without preceding B- tags for tinybert-nocrf: 170


In [152]:
df_i_without_b = pd.DataFrame.from_dict(i_without_b, orient="index", columns=["I_without_B"])
df_i_without_b = df_i_without_b.reset_index().rename(columns={"index": "model"})
df_i_without_b

,model,I_without_B
0,bilstm-crf,84
1,bilstm-nocrf,213
2,distilbert-crf,119
3,distilbert-nocrf,96
4,teacher-crf,127
5,teacher-nocrf,139
6,tinybert-crf,177
7,tinybert-nocrf,170


In [ ]:
# df_i_without_b.to_csv(r"D:\Dafa\Project\queryner-kd\result-data\entity-based-eval\i_without_b.csv", index=False)

### Classification Report

In [173]:
report = classification_report(gold_labels, pred_labels["teacher-crf"], output_dict=True, mode="strict", scheme=IOB2)

c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# reports_path = r"D:\Dafa\Project\queryner-kd\result-data\entity-based-eval\reports"

# for model in models_name_without_pt:
#     report = classification_report(gold_labels, pred_labels[model], output_dict=True, mode="strict", scheme=IOB2)
#     report_df = pd.DataFrame(report).transpose()
#     report_df = report_df.reset_index().rename(columns={"index": "label"})
#     report_df.to_csv(os.path.join(reports_path, f"{model}_report.csv"), index=False)

c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\seqeval\metrics\v1.py:57: